In [1]:
#!/usr/bin/env python3
import numpy as np, time, os
from polyhedron_gravitation_MT import PolyhedronGravitation

# --- robust path setup ---
try:
    ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    # running interactively (no __file__)
    # assume current working directory is Polyhedron/python
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA = os.path.join(ROOT, "data")
os.makedirs(DATA, exist_ok=True)

In [3]:
# ------------------------- load geometry -------------------------
verts_path = os.path.join(DATA, "icosahedron_vertices.csv")
faces_path = os.path.join(DATA, "icosahedron_faces.csv")
points_path = os.path.join(DATA, "eval_points_100k_plus_vertices.csv")

V = np.loadtxt(verts_path, delimiter=",")
F = np.loadtxt(faces_path, delimiter=",", dtype=int)
Pts = np.loadtxt(points_path, delimiter=",")

# convert to 0-based if needed
if np.min(F) == 1:
    F -= 1

print("\n--- Python benchmark (Acceleration + Tensor) ---")
print(f"Data folder: {DATA}")
print(f"Vertices: {V.shape[0]}, Faces: {F.shape[0]}, Points: {Pts.shape[0]}")

# ------------------------- model -------------------------
model = PolyhedronGravitation(V, F, G=1.0, density=1.0, eps=0.0, orient_faces=True)


# ------------------------- helper -------------------------
def save_time(path, total_s, npts):
    with open(path, "w") as f:
        f.write(f"total_time_sec: {total_s:.6f}\n")
        f.write(f"time_per_point_sec: {total_s / npts:.12e}\n")
        f.write(f"time_per_point_us: {(total_s / npts) * 1e6:.6f}\n")

# ======================================================================
#                     Acceleration benchmark
# ======================================================================
print("\n[Acceleration] Benchmarking...")
t0 = time.time()
A_py = model.acceleration(Pts, block_size=8192)
t_elapsed_accel = time.time() - t0
tpp_accel = t_elapsed_accel / len(Pts)

print(f"Acceleration time: {t_elapsed_accel:.3f} s | per-pt: {tpp_accel*1e6:.3f} µs")

np.savetxt(os.path.join(DATA, "A_python.csv"), A_py, delimiter=",")
save_time(os.path.join(DATA, "time_accel_python.txt"), t_elapsed_accel, len(Pts))

# ======================================================================
#                     Tensor benchmark
# ======================================================================
print("\n[Tensor] Benchmarking...")
t0 = time.time()
T_py = model.gravity_tensor(Pts, block_size=4096)
t_elapsed_tensor = time.time() - t0
tpp_tensor = t_elapsed_tensor / len(Pts)

print(f"Tensor time: {t_elapsed_tensor:.3f} s | per-pt: {tpp_tensor*1e6:.3f} µs")

# Flatten each tensor row (9 columns)
T_flat = T_py.reshape(len(Pts), 9)
np.savetxt(os.path.join(DATA, "T_python.csv"), T_flat, delimiter=",")
save_time(os.path.join(DATA, "time_tensor_python.txt"), t_elapsed_tensor, len(Pts))

print("\nResults saved:")
print("  A_python.csv + time_accel_python.txt")
print("  T_python.csv + time_tensor_python.txt")
print("--------------------------------------------------------------")



--- Python benchmark (Acceleration + Tensor) ---
Data folder: /Volumes/Dunendran/Programs/Banchmark/Polyhedron/data
Vertices: 12, Faces: 20, Points: 100012

[Acceleration] Benchmarking...
Acceleration time: 0.261 s | per-pt: 2.607 µs

[Tensor] Benchmarking...
Tensor time: 0.141 s | per-pt: 1.415 µs

Results saved:
  A_python.csv + time_accel_python.txt
  T_python.csv + time_tensor_python.txt
--------------------------------------------------------------
